# Tool & Artifact Contract (v1)

## Purpose
A tool is a small, declarative capability that reads/writes **artifacts** via a shared registry. This contract standardizes:
- how tools declare inputs/outputs and metadata,
- how they run,
- how artifacts are shaped/remembered,
- how planners propose and chain tool calls (action JSON).

---

## 1) Core Interfaces


### 1.1 Base Tool (shape)
A tool is a class with static metadata and a `run(...)` method.

- **TOOL_NAME**: snake_case id  
- **DESCRIPTION**: ALL CAPS one-liner  
- **CATEGORY**: one of `import | transform | analysis | export | display | management | chat | control`  
- **USAGE** (optional): “When to use” guidance (1–2 sentences)  
- **EXAMPLES** (optional): list of `{ "input": {...}, "why": "..." }`  
- **ARTIFACTS**: dict of artifact type declarations (see §2.2)  
- **IO_SCHEMA**: explicit inputs/outputs schema (see §1.2)  
- **run(input_data, artifacts, package_name=None, **kwargs) → dict**

### 1.2 IO_SCHEMA (what planners must follow)
```json
{
  "inputs": {
    "filename": { "type": "path", "required": true,  "description": "CSV file to import" },
    "top_n":    { "type": "integer", "required": false, "description": "Max results" }
  },
  "outputs": {
    "table_artifact_id": { "type": "table", "remember": false, "description": "Created artifact id" }
  }
}
```
Allowed `type` values (non-exhaustive): `string | integer | number | boolean | dict | list | path | table | hierarchy | capella_model | capella_selection | conversation | notebook | (custom)`  
- `required` (inputs): must be present in action JSON  
- `remember` (outputs): if `true`, agent may surface in conversation memory Meaning
-- When an output field sets remember: true, the tool promises to return a stable artifact ID for that output. The agent will:
-- tag the artifact as remembered in the artifact registry (artifact memory), and
-- add a small pointer (id + note) to the agent’s conversation memory (scoped by package).
-- Why
-- So downstream prompts can say “recall the hierarchy we just imported” without reloading/retreading work.

### 1.3 Tool output (recommended fields)
- `message`: short human summary  
- `artifact_ids`: map from IO_SCHEMA output keys → artifact IDs  
- `content`: raw payload (display tools)  
- `ui`/`html`: optional preview text/HTML  
- `error`: string on failure (plus optional `message`)

---

## 2) Artifacts

### 2.1 Artifact record (in registry)
```json
{
  "id": "uuid",
  "type": "table",
  "name": "optional-friendly-name",
  "content": { },
  "metadata": { },
  "_created_at": "ISO-8601"
}
```

### 2.2 Declaring artifact types (on tools)
```json
{
  "capella_model": {
    "schema_version": "1.0",
    "description": "Bundled Capella model reference (path + resources).",
    "fields": {
      "path":      { "type": "path" },
      "resources": { "type": "dict" }
    }
  }
}
```

### 2.3 Announce (optional)
A tool may set a one-shot banner on new artifacts (e.g., "✅ Artifact created: …"). Agents may display it once.

---

## 3) Planner Action JSON (what the AI emits)

### 3.1 Format (ONLY this)
```json
{"actions":[
  {"tool":"read_leveled_csv","input":{"filename":"drone.csv"}},
  {"tool":"name_artifact","input":{"type":"hierarchy","name":"BOM"}}
]}
```

### 3.2 Hard rules for planners
- **Use ONLY fields** declared in IO_SCHEMA.inputs.  
- **Never invent** fields (e.g., `version` unless defined).  
- **Omit empty** values (`"", null`).  
- **Satisfy all required** inputs; if missing, add actions to create/fetch prerequisites first.  
- Reference artifacts as specified (e.g., `capella_model_name` **or** `_id`, not both unless required).  
- Don’t include `package` unless declared by the tool.  
- Keep plans minimal (≤3 actions unless truly needed).  
- If no tool applies, respond naturally (no JSON).  
- **Do NOT prefix with `json` or code fences**—return raw JSON starting with `{` and ending with `}`.

---

## 4) Tool Development Checklist
1. Pick **TOOL_NAME**, **DESCRIPTION**, **CATEGORY**.  
2. If introducing a new artifact type, declare it in **ARTIFACTS**.  
3. Define **IO_SCHEMA** (inputs/outputs; set `remember` as needed).  
4. Implement `run(...)`: validate → do work → create artifacts via `artifacts.add_artifact(...)` → return dict (`message`, `artifact_ids`, etc.).  
5. Add **USAGE** and optional **EXAMPLES** to help planning.  
6. Register the tool (decorator or registry call).

---

## 5) Canonical Examples (concise)

### 5.1 Import leveled CSV → hierarchy
```python

from se_agent.core.tool_patterns import register_tool, ImportTool

@register_tool
class ReadLeveledCSVTool(ImportTool):
    TOOL_NAME="read_leveled_csv"; DESCRIPTION="IMPORTS A LEVELED CSV FILE AND CREATES A HIERARCHY ARTIFACT."; CATEGORY="import"
    USAGE="Use to ingest a CSV with hierarchical columns (Level1..LevelN)."
    ARTIFACTS={"hierarchy":{"schema_version":"1.0","description":"Hierarchy from leveled CSV","fields":{"levels":{"type":"list"},"records":{"type":"list"},"source_file":{"type":"path"}}}}
    IO_SCHEMA={"inputs":{"filename":{"type":"path","required":true}},"outputs":{"hierarchy_artifact_id":{"type":"hierarchy","remember":true}}}
    def run(self,input_data,artifacts,package_name=None,**_):
        import csv; fn=input_data["filename"]
        with open(fn,newline="",encoding="utf-8") as f:
            r=csv.DictReader(f); levels=list(r.fieldnames or []); rows=[dict(x) for x in r]
        art=artifacts.add_artifact(package_name,"hierarchy",{"levels":levels,"records":rows,"source_file":fn},{"rows":len(rows),"source":fn})
        return {"message":f"✅ Imported '{fn}' into hierarchy.","artifact_ids":{"hierarchy_artifact_id":art.id}}
```

### 5.2 Create Capella model artifact (single source of truth)
```python
from se_agent.core.tool_patterns import register_tool, ImportTool

@register_tool
class CreateCapellaModelArtifactTool(ImportTool):
    TOOL_NAME="create_capella_model_artifact"; DESCRIPTION="CREATE A SINGLE CAPELLA_MODEL ARTIFACT FROM PATH AND RESOURCES."; CATEGORY="import"
    USAGE="Use once per project to register the .aird path + resources."
    ARTIFACTS={"capella_model":{"schema_version":"1.0","description":"Bundled Capella model reference","fields":{"path":{"type":"path"},"resources":{"type":"dict"}}}}
    IO_SCHEMA={"inputs":{"path_to_model":{"type":"path","required":true},"resources":{"type":"dict","required":true},"name":{"type":"string"}},"outputs":{"capella_model_artifact_id":{"type":"capella_model","remember":true}}}
    def run(self,input_data,artifacts,package_name=None,**_):
        content={"path":input_data["path_to_model"],"resources":input_data["resources"]}; meta={}
        art=artifacts.add_artifact(package_name,"capella_model",content,meta)
        if input_data.get("name"): art.name=input_data["name"]
        return {"message":f"✅ Created capella_model from '{content['path']}'.","artifact_ids":{"capella_model_artifact_id":art.id}}
```

### 5.3 Query Capella (embeddings) → capella_selection
```python
from se_agent.core.tool_patterns import register_tool, TransformTool

@register_tool
class QueryCapellaModelTool(TransformTool):
    TOOL_NAME="query_capella_model"; DESCRIPTION="QUERY A CAPELLA MODEL USING EMBEDDINGS AND SAVE A SELECTION ARTIFACT."; CATEGORY="analysis"
    USAGE="Use after a capella_model exists; returns top matches for text."
    ARTIFACTS={"capella_selection":{"schema_version":"1.0","description":"Matched elements via embeddings","fields":{"query":{"type":"string"},"model_path":{"type":"path"},"count":{"type":"integer"},"matches":{"type":"list"},"embedding_file":{"type":"path"}}}}
    IO_SCHEMA={"inputs":{"capella_model_name":{"type":"string"},"capella_model_id":{"type":"string"},"query":{"type":"string","required":true},"top_n":{"type":"integer"},"embedding_file":{"type":"path"}},"outputs":{"selection_artifact_id":{"type":"capella_selection","remember":true}}}
    def run(self,input_data,artifacts,package_name=None,**_):
        from pathlib import Path; import hashlib
        import capellambse; from capella_tools import capella_embeddings_manager as CEM
        pkg=artifacts.get_package(package_name)
        cm=None
        if input_data.get("capella_model_name"):
            c=[a for a in pkg.artifacts.values() if a.name==input_data["capella_model_name"] and a.type=="capella_model"]
            if not c: return {"error":f"capella_model '{input_data['capella_model_name']}' not found."}
            cm=sorted(c,key=lambda a:getattr(a,"_created_at",""),reverse=True)[0]
        elif input_data.get("capella_model_id"):
            cm=pkg.artifacts.get(input_data["capella_model_id"])
            if not cm or cm.type!="capella_model": return {"error":"Invalid capella_model_id."}
        else: return {"error":"Provide capella_model_name or capella_model_id."}
        path_to_model=cm.content["path"]; resources=cm.content["resources"]; query=input_data["query"]; top_n=int(input_data.get("top_n",50))
        m=capellambse.MelodyModel(path_to_model,resources=resources)
        mgr=CEM.EmbeddingManager(); emb=input_data.get("embedding_file")
        if not emb:
            sha8=hashlib.sha1((mgr.model or "").encode()).hexdigest()[:8]
            emb=f"{Path(path_to_model).stem}.embeddings.{sha8}.json"
        mgr.set_files(path_to_model,emb); mgr.create_model_embeddings(m)
        selected=mgr.query_and_select_top_objects(query,top_n=top_n) or []
        out=[{"uuid":str(getattr(o,"uuid","") or getattr(o,"id","") or ""), "name":str(getattr(o,"name","") or ""), "type":type(o).__name__} for o in selected]
        sel=artifacts.add_artifact(package_name,"capella_selection",{"query":query,"model_path":path_to_model,"count":len(out),"matches":out,"embedding_file":emb},{"source":"query_capella_model","top_n":top_n})
        return {"message":f"✅ Query completed: {len(out)} match(es) saved in selection '{sel.id[:8]}'","artifact_ids":{"selection_artifact_id":sel.id}}
```

### 5.4 Display raw artifact
```python
from se_agent.core.tool_patterns import register_tool, DisplayTool

@register_tool
class ShowArtifactTool(DisplayTool):
    TOOL_NAME="show_artifact"; DESCRIPTION="DISPLAY A STORED ARTIFACT (RAW CONTENT) BY ID, NAME, OR TYPE."; CATEGORY="display"
    USAGE="Use to inspect a specific artifact."
    IO_SCHEMA={"inputs":{"id":{"type":"string"},"name":{"type":"string"},"type":{"type":"string"}},"outputs":{}}
    def run(self,input_data,artifacts,package_name=None,**_):
        pkg=artifacts.get_package(package_name); art=None
        if input_data.get("id"): art=pkg.artifacts.get(input_data["id"])
        elif input_data.get("name"):
            c=[a for a in pkg.artifacts.values() if a.name==input_data["name"]]; art=sorted(c,key=lambda a:getattr(a,"_created_at",""),reverse=True)[0] if c else None
        elif input_data.get("type"):
            c=[a for a in pkg.artifacts.values() if a.type==input_data["type"]]; art=sorted(c,key=lambda a:getattr(a,"_created_at",""),reverse=True)[0] if c else None
        if not art: return {"error":"Artifact not found."}
        return {"message":f"Artifact: {art.name or art.id}","artifact_id":art.id,"type":art.type,"name":art.name,"metadata":art.metadata or {},"content":art.content}
```

---

## 6) Planner Contract (paste at top of the context)

You are a systems engineering assistant with access to tools.  
Before proposing actions, consult each tool’s IO_SCHEMA below.

When actions can be taken, reply **ONLY** with a JSON object containing an `actions` list.  
Each action: `{"tool": <tool_name>, "input": {<fields>}}`.

**INPUT RULES**
- Include **ONLY** inputs defined in the tool’s IO_SCHEMA.inputs.  
- **NEVER invent** fields; omit empty values (`"", null`).  
- Satisfy **all required** inputs; create/fetch prerequisites first.  
- Use artifact references as specified (e.g., `capella_model_name` **or** `capella_model_id`).  
- Do not include `package` unless the tool declares it.  
- Match input types (`string`, `integer`, `boolean`, `dict`, `list`, `path`).  

**PLANNING**
- Chain tools using declared outputs; keep sequences minimal (≤3).  
- If no tool applies or required data cannot be created, reply naturally (no JSON).  

**FORMAT**
- Return **ONLY** the JSON with `actions`. No code fences, no `json` prefix, no commentary.  
- Do not propose `{"tool":"interactive_chat"}` recursively.  

**EXAMPLES**
```json
{"actions":[{"tool":"read_leveled_csv","input":{"filename":"drone.csv"}}]}
{"actions":[{"tool":"name_artifact","input":{"type":"hierarchy","name":"BOM"}}]}
{"actions":[{"tool":"show_artifact","input":{"name":"SEA_Capella_Model"}}]}
{"actions":[{"tool":"query_capella_model","input":{"capella_model_name":"BikeModel","query":"brake lever","top_n":25}}]}
```

If no tool applies, respond naturally.


Example of run command.



In [1]:
agent.run('create_capella_model_artifact', input_data=
          {'path_to_model': '/home/simcenter/studio/SE_Agent/SE_Agent/SE_Agent.aird',
           'resources': {'SE_Agent': '/home/simcenter/studio/SE_Agent/SE_Agent/'}})

NameError: name 'agent' is not defined

# How to Invoke Tools via the Agent

This reference table shows how to invoke each tool using the `agent.run` format, along with their expected behavior and typical usage flow.

| Tool Name                      | Purpose                                                                                               | Example `agent.run` Invocation                                                                                                                                                                                                                                                                       | Expected Result                                                                                     |
| ------------------------------ | ----------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------- |
| **load_prompt_path**           | Registers a directory containing notebook prompts as a `prompt_path` artifact.                        | `agent.run("load_prompt_path", {"prompt_dir_path":"/home/simcenter/prompts/icd_compare","name":"My_Prompts_prompt_path"})`                                                                                                                                                                           | ✅ Creates artifact `prompt_path` with fields: `directory_path`, `notebooks`, `count`, `scanned_at`. |
| **search_prompts**             | Lists available notebooks inside a `prompt_path` artifact (optionally filtered).                      | `agent.run("search_prompts", {"prompt_path_name":"My_Prompts_prompt_path","query":"icd"})`                                                                                                                                                                                                           | 📋 Returns a list of `.ipynb` files under the directory, optionally filtered by substring.          |
| **build_prompt_from_notebook** | Executes a notebook (from a `prompt_path`) that emits prompt JSON → creates a `prompt_spec` artifact. | `agent.run("build_prompt_from_notebook", {"prompt_path_name":"My_Prompts_prompt_path","prompt_name":"icd_compare.ipynb","kernel_name":"python3"})`                                                                                                                                                   | ⚙️ Runs notebook, captures JSON from a `prompt_json` cell, and creates a `prompt_spec` artifact.    |
| **show_prompt_spec**           | Displays a built `prompt_spec` artifact in readable form.                                             | `agent.run("show_prompt_spec", {"prompt_spec_name":"icd_component_compare_spec"})`                                                                                                                                                                                                                   | 🪶 Outputs header (Prompt title/key/version) and a pretty JSON preview.                             |
| **render_prompt**              | Renders the prompt template with given variables or positional args.                                  | Variables form: `agent.run("render_prompt", {"prompt_spec_name":"icd_component_compare_spec","variables":{"a":"CompA","b":"CompB","ruleset":"default"}})`<br><br>Positional form: `agent.run("render_prompt", {"prompt_spec_name":"icd_component_compare_spec","args":["CompA","CompB","default"]})` | ✨ Returns `rendered` (final text), `resolved` (variable map), and a `content_preview`.              |
| **generate_capella_fabric**    | Converts selected Capella elements (via `capella_selection`) into a YAML “fabric” artifact.           | `agent.run("generate_capella_fabric", {"capella_model_name":"SEA_Capella_Model","selection_name":"CS"})`                                                                                                                                                                                             | 🧩 Creates a `capella_fabric` artifact containing YAML, targets, and metadata from the model.       |
| **show_artifact**              | Displays any artifact by name or id.                                                                  | `agent.run("show_artifact", {"name":"CS"})`                                                                                                                                                                                                                                                          | 📦 Prints artifact metadata and content preview (truncated for long text).                          |

---

## Typical Workflow Example

1. **Register** your prompt notebooks → `load_prompt_path`
2. **Browse** available prompt notebooks → `search_prompts`
3. **Build** a structured prompt spec → `build_prompt_from_notebook`
4. **Inspect** or verify the spec → `show_prompt_spec`
5. **Render** the prompt with live variables → `render_prompt`
6. (Optional) **Use rendered text** as input for downstream tools or agents.


### 🧩 Tool UI Output Behavior

Each tool can return structured display fields to control how its results appear in the agent interface or notebook:

| Field | Type | Purpose | Renderer behavior |
|-------|------|----------|-------------------|
| `message` | `string` | A concise summary or status line. | Displayed *only* if no richer `ui` or `html` content is present. |
| `ui` | `string` (Markdown) | The primary user-facing Markdown view — used for tables, bullet summaries, or formatted reports. | Rendered first (preferred). |
| `html` | `string` (HTML) | A richer version of the same content, for embedding in web or notebook environments. | Rendered if no `ui` is provided. |
| `displayed` | `boolean` | If `True`, the renderer assumes the tool already displayed its own output. | Prevents `_show_tool_result` from re-rendering. |
| `artifacts` / `memory` | `list[dict]` | Structured data (for list tools). | Shown only if no `ui` or `html` is returned. |
| `inject_once` | `string` | Temporary one-shot prompt content. | Shown as a note if present. |

#### Rendering Priority
1. `ui` (Markdown)
2. `html`
3. `artifacts` / `memory` tables
4. `message`
5. Fallback (`text`, `csv_text`, etc.)

This ensures that tools producing rich Markdown (like `list_workspace` or `format_json_report`) show a single clean view, while simpler tools (e.g., file writers) show just their short status message.

#### Notes for Tool Authors
- For list/report tools: populate both `message` (short summary) and `ui` (table Markdown).
- For visualization tools: return `html` or set `displayed=True` if you handle your own rendering.
- For headless data transformers: return only `message` and any outputs in `outputs`.
- Use icons consistently for memory scope:  
  ⚙️ Working 📋 Short-Term 🗄️ Long-Term.
